# Multi-task DCVC-RT-VCM · Bước 4 + đo thời gian train

Notebook thử nghiệm (nhánh `multitask-exp` của `uetot1/bla`), **không đụng** pipeline RIVF cũ.

Làm 3 việc:
1. **Self-check** môi trường Kaggle (λ 1–64, nhánh seg đóng băng vẫn truyền gradient về codec, checkpoint đọc được bằng `evaluate_vcm.py`).
2. **Bước 4 – `measure_task_scales`** trên 200 clip Vimeo: ra `seg_scale` (cân D_seg về thang D_det) và CKA theo từng layer giữa 2 teacher — xác nhận lại lựa chọn layer 17.
3. **Chạy thử 100 batch** trên T4 ×2 → tự đổi ra thời gian 1 epoch và 1 run 10 epoch.

**Settings > Accelerator:** GPU T4 x2 · **Internet:** On (để clone repo + tải `yolov5s-seg.pt`)

**Attach vào `/kaggle/input`:**
- Vimeo-90K Septuplet (có `sequences/`, `sep_trainlist.txt`, `sep_testlist.txt`)
- `cvpr2025_image.pth.tar`, `cvpr2025_video.pth.tar`
- Checkpoint bài RIVF (λ 1–64) — notebook tự tìm, hoặc đặt tay `INIT_CHECKPOINT`

In [ ]:
from pathlib import Path
import json
import torch

assert torch.cuda.is_available(), 'Hay bat GPU trong Kaggle Settings > Accelerator'
GPU_COUNT = torch.cuda.device_count()
assert GPU_COUNT >= 2, f'Can GPU T4 x2, hien chi thay {GPU_COUNT}.'
for index in range(GPU_COUNT):
    print(f'GPU {index}:', torch.cuda.get_device_name(index))

KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
OUTPUT_DIR = KAGGLE_WORKING / 'multitask_step4'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Code ---------------------------------------------------------------------------
REPO = 'https://github.com/uetot1/bla.git'
BRANCH = 'multitask-exp'
COMMIT = '5d5428df39d2579cddba284e94150d4d228600b3'

# ---- Checkpoint bai RIVF (warm start) ------------------------------------------------
# None = tu tim trong /kaggle/input: chi nhan checkpoint co lambda_mapping='geometric_qp',
# lambda_range=(1, 64) va co cloned_frontend_state_dict. Nhieu file thi dat tay duong dan.
INIT_CHECKPOINT = None

# ---- Buoc 4 -------------------------------------------------------------------------
MEASURE_CLIPS = 200
MEASURE_QPS = '0,21,42,63'
SEG_LAYER = 17
CKA_LAYERS = '4,6,9,13,17,20,23'

# ---- Chay thu do thoi gian (giu dung cau hinh train bai RIVF) ------------------------
NPROC_PER_NODE = 2
GLOBAL_BATCH_SIZE = 4
LEARNING_RATE = 1e-6
GRAD_CLIP = 1.0
GROUP_SIZE = 6
WORKERS = 4
ALPHA_DET = 0.5          # R2
TIMING_BATCHES = 100
TIMING_VAL_BATCHES = 10
PLANNED_EPOCHS = 10
PLANNED_VAL_BATCHES = 250

YOLOV5S_SHA256 = '8b3b748c1e592ddd8868022e8732fde20025197328490623cc16c6f24d0782ee'
YOLOV5S_SEG_SHA256 = '78558867515e9d9654c52ed87c40c22d90a983bea63450c611cbd8c2b13e3aa2'

In [ ]:
# Clone dung commit + build entropy coder C++ (giong notebook train bai RIVF).
import os
import subprocess
import sys

PROJECT = KAGGLE_WORKING / 'svc_multitask'
if not (PROJECT / '.git').is_dir():
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO, str(PROJECT)], check=True)
subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=PROJECT, check=True)
subprocess.run(['git', 'checkout', '--detach', COMMIT], cwd=PROJECT, check=True)
assert (PROJECT / 'multitask_exp' / 'train_multitask.py').is_file(), 'Sai commit: khong co multitask_exp'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'setuptools<81'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'requirements.txt')], check=True)

setup_py = PROJECT / 'dcvc_rt/src/cpp/setup.py'
text = setup_py.read_text(encoding='utf-8').replace('python_requires=">=3.12"', 'python_requires=">=3.10"')
setup_py.write_text(text, encoding='utf-8')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(setup_py.parent)], check=True)

os.chdir(PROJECT)
import MLCodec_extensions_cpp  # noqa: F401
print('Project:', PROJECT, '| commit', COMMIT[:7])
print('Entropy coder extension: OK')

In [ ]:
# Weight YOLO: yolov5s.pt co san trong repo; yolov5s-seg.pt tai tu release v7.0.
import hashlib
import urllib.request


def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1 << 20), b''):
            digest.update(chunk)
    return digest.hexdigest()


DET_WEIGHTS = PROJECT / 'yolov5s.pt'
SEG_WEIGHTS = PROJECT / 'multitask_exp' / 'weights' / 'yolov5s-seg.pt'
SEG_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
if not SEG_WEIGHTS.is_file():
    urllib.request.urlretrieve(
        'https://github.com/ultralytics/yolov5/releases/download/v7.0/yolov5s-seg.pt', SEG_WEIGHTS)

assert sha256(DET_WEIGHTS) == YOLOV5S_SHA256, 'yolov5s.pt khac weight da dung trong bai'
assert sha256(SEG_WEIGHTS) == YOLOV5S_SEG_SHA256, 'yolov5s-seg.pt tai ve bi sai/hong'
print('YOLO weights OK:', DET_WEIGHTS.name, SEG_WEIGHTS.name)

In [ ]:
# Tim Vimeo, DCVC-RT goc va checkpoint bai RIVF (lambda 1-64).
def unique_input_file(filename):
    matches = list(KAGGLE_INPUT.rglob(filename))
    assert len(matches) == 1, f'Can dung 1 file {filename}, tim thay: {matches}'
    return matches[0]


vimeo_candidates = [
    path.parent for path in KAGGLE_INPUT.rglob('sep_trainlist.txt')
    if (path.parent / 'sep_testlist.txt').is_file() and (path.parent / 'sequences').is_dir()
]
assert len(vimeo_candidates) == 1, f'Can dung 1 thu muc Vimeo septuplet, thay: {vimeo_candidates}'
VIMEO_ROOT = vimeo_candidates[0]
IMAGE_CKPT = unique_input_file('cvpr2025_image.pth.tar')
VIDEO_CKPT = unique_input_file('cvpr2025_video.pth.tar')


def describe_checkpoint(path):
    try:
        checkpoint = torch.load(path, map_location='cpu', weights_only=True)
    except Exception as error:
        return False, f'khong doc duoc ({type(error).__name__})'
    if not isinstance(checkpoint, dict) or 'state_dict' not in checkpoint:
        return False, 'khong co state_dict'
    mapping = checkpoint.get('lambda_mapping')
    lambda_range = tuple(checkpoint.get('lambda_range', ()))
    has_clone = 'cloned_frontend_state_dict' in checkpoint
    info = f"mapping={mapping} range={lambda_range} epoch={checkpoint.get('epoch')} clone={has_clone}"
    ok = mapping == 'geometric_qp' and lambda_range == (1.0, 64.0) and has_clone
    return ok, info


if INIT_CHECKPOINT is None:
    candidates = []
    for path in sorted(list(KAGGLE_INPUT.rglob('*.pth')) + list(KAGGLE_INPUT.rglob('*.pth.tar'))):
        if path.name.startswith('cvpr2025_'):
            continue
        ok, info = describe_checkpoint(path)
        print(('  [NHAN] ' if ok else '  [bo]   ') + str(path), '->', info)
        if ok:
            candidates.append(path)
    assert len(candidates) == 1, (
        f'Can dung 1 checkpoint lambda 1-64, tim thay {len(candidates)}. '
        'Dat tay INIT_CHECKPOINT o cell cau hinh.')
    INIT_CHECKPOINT = candidates[0]
else:
    INIT_CHECKPOINT = Path(INIT_CHECKPOINT)
    ok, info = describe_checkpoint(INIT_CHECKPOINT)
    assert ok, f'INIT_CHECKPOINT khong phai checkpoint lambda 1-64 co clone: {info}'

print()
print('VIMEO_ROOT      :', VIMEO_ROOT)
print('IMAGE_CKPT      :', IMAGE_CKPT)
print('VIDEO_CKPT      :', VIDEO_CKPT)
print('INIT_CHECKPOINT :', INIT_CHECKPOINT, '| sha256', sha256(INIT_CHECKPOINT)[:16])

In [ ]:
from collections import deque


def run_and_stream(command, tail=60):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(
        command, cwd=PROJECT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1)
    last = deque(maxlen=tail)
    for line in process.stdout:
        last.append(line)
        if 'pkg_resources' not in line:
            print(line, end='', flush=True)
    exit_code = process.wait()
    assert exit_code == 0, f'Lenh that bai (exit {exit_code}). Log cuoi:\n' + ''.join(last)


# 1) Self-check tren moi truong Kaggle
run_and_stream([sys.executable, '-m', 'multitask_exp.train_multitask', '--self_check',
                '--det_weights', DET_WEIGHTS, '--seg_weights', SEG_WEIGHTS,
                '--save_dir', OUTPUT_DIR / 'self_check'])

In [ ]:
# 2) Buoc 4: seg_scale + CKA theo layer tren Vimeo that
TASK_SCALES = OUTPUT_DIR / 'task_scales.json'
run_and_stream([
    sys.executable, '-m', 'multitask_exp.measure_task_scales',
    '--dataset', VIMEO_ROOT, '--model_path_i', IMAGE_CKPT, '--model_path_p', VIDEO_CKPT,
    '--init_checkpoint', INIT_CHECKPOINT, '--det_weights', DET_WEIGHTS, '--seg_weights', SEG_WEIGHTS,
    '--seg_layer', SEG_LAYER, '--clips', MEASURE_CLIPS, '--qps', MEASURE_QPS,
    '--group_size', GROUP_SIZE, '--cka_layers', CKA_LAYERS, '--json_out', TASK_SCALES,
], tail=200)

scales = json.loads(TASK_SCALES.read_text())
print()
print(f"seg_scale = {scales['seg_scale']:.6g}   (clips={scales['clips']}, det clone tu {scales['det_clone_source']})")
print(f"{'base_qp':>7} {'D_det':>11} {'D_seg':>11} {'ratio':>10} {'bpp':>8}")
for row in scales['per_qp']:
    print(f"{row['base_qp']:>7} {row['mean_d_det']:>11.5g} {row['mean_d_seg']:>11.5g} "
          f"{row['d_det_over_d_seg']:>10.4g} {row['mean_bpp']:>8.4f}")

print()
print(f"CKA tren {scales['cka_images']} frame Vimeo goc")
print(f"{'layer':>5} {'det_vs_seg':>11} {'det_vs_blur':>12} {'khoang_cach':>12}")
gaps = {}
for row in scales['cka_by_layer']:
    gap = row['cka_det_vs_det_blurred'] - row['cka_det_vs_seg']
    gaps[row['layer']] = gap
    print(f"{row['layer']:>5} {row['cka_det_vs_seg']:>11.3f} {row['cka_det_vs_det_blurred']:>12.3f} {gap:>12.3f}")
print()
print('Layer hai teacher khac nhau ro nhat:', max(gaps, key=gaps.get))
print(f"Khoang cach layer 4 = {gaps.get(4, float('nan')):.3f} | layer {SEG_LAYER} = {gaps.get(SEG_LAYER, float('nan')):.3f}")
print('-> Giu layer', SEG_LAYER, 'neu khoang cach cua no lon hon ro so voi layer 4.')

In [ ]:
# 3) Chay thu 100 batch tren T4 x2 (R2, alpha_det=0.5) de do thoi gian
TIMING_DIR = OUTPUT_DIR / 'runs'
run_and_stream([
    sys.executable, '-m', 'torch.distributed.run', f'--nproc_per_node={NPROC_PER_NODE}',
    '--module', 'multitask_exp.train_multitask',
    '--dataset', VIMEO_ROOT, '--model_path_i', IMAGE_CKPT, '--model_path_p', VIDEO_CKPT,
    '--init_checkpoint', INIT_CHECKPOINT, '--det_weights', DET_WEIGHTS, '--seg_weights', SEG_WEIGHTS,
    '--seg_layer', SEG_LAYER, '--alpha_det', ALPHA_DET, '--task_scale_file', TASK_SCALES,
    '--batch_size', GLOBAL_BATCH_SIZE, '--learning_rate', LEARNING_RATE, '--grad_clip', GRAD_CLIP,
    '--group_size', GROUP_SIZE, '--workers', WORKERS,
    '--epochs', 1, '--max_train_batches', TIMING_BATCHES, '--max_val_batches', TIMING_VAL_BATCHES,
    '--save_dir', TIMING_DIR, '--run_name', 'timing',
], tail=120)

In [ ]:
record = json.loads((TIMING_DIR / 'timing' / 'training_history.json').read_text())[-1]
seconds_per_step = record['seconds_per_global_batch']
steps_per_epoch = record['full_epoch_batches']
epoch_hours = seconds_per_step * steps_per_epoch / 3600
# Validation chi forward, so batch nho -> uoc luong tho bang 1/3 chi phi 1 buoc train.
val_hours = seconds_per_step * PLANNED_VAL_BATCHES / 3 / 3600
run_hours = PLANNED_EPOCHS * (epoch_hours + val_hours)

summary = {
    'commit': COMMIT,
    'init_checkpoint': str(INIT_CHECKPOINT),
    'seg_scale': scales['seg_scale'],
    'seconds_per_global_step': seconds_per_step,
    'steps_per_epoch': steps_per_epoch,
    'estimated_epoch_hours': epoch_hours,
    'estimated_run_hours_for_planned_epochs': run_hours,
    'planned_epochs': PLANNED_EPOCHS,
    'timing_batches': TIMING_BATCHES,
    'train_loss_first_batches': record['total_loss'],
    'feature_mse_det': record['feature_mse_det'],
    'feature_mse_seg': record['feature_mse_seg'],
}
(OUTPUT_DIR / 'step4_summary.json').write_text(json.dumps(summary, indent=2))

print(f'{seconds_per_step:.3f} s / buoc (tong batch {GLOBAL_BATCH_SIZE}, T4 x{NPROC_PER_NODE})')
print(f'{steps_per_epoch} buoc / epoch  ->  ~{epoch_hours * 60:.0f} phut / epoch')
print(f'1 run {PLANNED_EPOCHS} epoch (kem validation {PLANNED_VAL_BATCHES} batch/epoch): ~{run_hours:.1f} gio')
print(f'3 run R2, R0, R1: ~{3 * run_hours:.1f} gio')
print('Luu y: 100 buoc dau gom ca warm-up nen so nay hoi bi quan.')
print()
print('Da luu:', OUTPUT_DIR / 'task_scales.json', 'va', OUTPUT_DIR / 'step4_summary.json')

## Sau khi chạy xong

Tải về (hoặc lưu thành Kaggle Dataset) thư mục `/kaggle/working/multitask_step4/`:
- `task_scales.json` — dùng làm `--task_scale_file` cho mọi run train.
- `step4_summary.json` — thời gian mỗi bước, mỗi epoch, mỗi run.

**Mốc G0 đạt khi:** self-check pass · CKA xác nhận layer 17 khác biệt rõ hơn layer 4 · biết thời gian 1 epoch.
Gửi lại 2 file JSON để chốt số epoch và bắt đầu train R2 → R0 → R1.